# Phase 7 — GARCH(1,1) VaR

Phase 4 showed that all three VaR models fail backtesting at 99%.
The root cause: static rolling windows weight every day equally,
so the model is blind to volatility clustering — bad days predict bad days.

GARCH(1,1) fixes this by modelling time-varying volatility explicitly:

    σ²_t = ω + α·ε²_{t-1} + β·σ²_{t-1}

where:
    ω = long-run variance (baseline)
    α = reaction to yesterday's shock (ARCH term)
    β = persistence of yesterday's variance (GARCH term)
    α + β < 1 ensures stationarity (variance mean-reverts)

VaR is then: -z_c · σ_t   (where σ_t is GARCH conditional vol on day t)

Because σ_t rises after a bad day, the VaR threshold automatically
widens — catching the clustering that Christoffersen penalised.

In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import norm, chi2
from arch import arch_model

sns.set_theme(style="darkgrid")

with open("portfolio_data.pkl", "rb") as f:
    data = pickle.load(f)

portfolio_returns = data["portfolio_returns"]
conf_levels       = data["conf_levels"]
lookback          = data["lookback"]

# Phase 4 rolling VaR results for direct comparison
all_bt     = data["backtest_all_results"]
kup_99     = data["kupiec_results_99"]
chris_99   = data["christoffersen_99"]

returns = portfolio_returns.values
dates   = portfolio_returns.index

print("Loaded successfully.")
print(f"Observations : {len(returns)}")
print(f"Date range   : {dates[0].date()} → {dates[-1].date()}")

Loaded successfully.
Observations : 2514
Date range   : 2015-01-05 → 2024-12-30


## Step 1 — Fit GARCH(1,1) on Full Sample
Fit on all 2514 days first to inspect the estimated parameters.
This tells us whether volatility clustering is statistically present.

In [ ]:
# arch_model expects returns in percentage terms (multiply by 100)
# rescaled_vol = actual_vol * 100, VaR rescaled back at the end
returns_pct = returns * 100

am = arch_model(
    returns_pct,
    vol="Garch",   # GARCH variance process
    p=1,           # lag order for ARCH term (ε²_{t-1})
    q=1,           # lag order for GARCH term (σ²_{t-1})
    mean="Zero",   # μ = 0 (same conservative assumption as Phase 2)
    dist="normal", # normal innovations — we'll compare to Student-t later
)

res = am.fit(disp="off")   # disp="off" suppresses optimiser output
print(res.summary())

omega = res.params["omega"]
alpha = res.params["alpha[1]"]
beta  = res.params["beta[1]"]

print(f"\n── Estimated Parameters ──────────────────────────────")
print(f"  ω (long-run baseline)  : {omega:.6f}")
print(f"  α (shock reaction)     : {alpha:.4f}")
print(f"  β (vol persistence)    : {beta:.4f}")
print(f"  α + β                  : {alpha + beta:.4f}  (must be < 1 for stationarity)")
print(f"  Long-run daily vol     : {np.sqrt(omega / (1 - alpha - beta)):.4f}%")

# What to look for:
# α + β close to 1 → high volatility persistence (typical for equity returns)
# Large β → today's vol is mostly yesterday's vol
# Large α → vol reacts strongly to shocks

## Step 2 — Plot Conditional Volatility vs Actual Returns
The GARCH conditional vol should spike during 2020 and 2022 —
showing it reacts to market stress, unlike the static rolling window.

In [ ]:
# conditional_volatility is the fitted σ_t series (in % terms)
cond_vol = res.conditional_volatility   # percentage units

fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)

# Panel 1: actual returns
axes[0].fill_between(dates, returns * 100, 0,
                     where=(returns >= 0), color="steelblue", alpha=0.6, label="Gain")
axes[0].fill_between(dates, returns * 100, 0,
                     where=(returns < 0),  color="tomato",    alpha=0.6, label="Loss")
axes[0].set_ylabel("Daily Return (%)")
axes[0].set_title("Portfolio Daily Returns")
axes[0].legend(fontsize=8)

# Panel 2: GARCH conditional volatility
axes[1].plot(dates, cond_vol, color="darkorange", linewidth=1.0,
             label="GARCH(1,1) Conditional Vol")
axes[1].set_ylabel("Conditional Volatility (%)")
axes[1].set_title("GARCH(1,1) Conditional Volatility — Spikes During Stress Periods")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nConditional Vol stats:")
print(f"  Min  : {cond_vol.min():.4f}%  (calmest day)")
print(f"  Max  : {cond_vol.max():.4f}%  (most volatile day)")
print(f"  Mean : {cond_vol.mean():.4f}%")

# What to look for:
# The orange line should spike sharply in March 2020 (COVID) and late 2022
# After each spike it decays gradually — that's mean-reversion (β controls the speed)
# Calm periods (2017, mid-2019) should show a flat, low baseline

## Step 3 — Rolling GARCH VaR (Out-of-Sample)
For a fair backtest we must use only PAST data to forecast each day's VaR.
We refit GARCH on a rolling 252-day window and use the 1-step-ahead
conditional volatility forecast as σ_t for that day's VaR.

    VaR_GARCH(t) = -z_c · σ_forecast(t)

This is the same rolling structure as Phase 4 — purely out-of-sample.

In [ ]:
def rolling_garch_var(
    returns: np.ndarray,
    lookback: int,
    confidence: float,
) -> np.ndarray:
    """
    Rolling GARCH(1,1) VaR.
    On each day t, fits GARCH on returns[t-lookback:t],
    takes the 1-step-ahead conditional vol forecast,
    and computes VaR = -z_c * σ_forecast.
    Returns VaR as a POSITIVE array (loss convention), NaN before warmup.
    """
    n   = len(returns)
    var = np.full(n, np.nan)
    z   = norm.ppf(1 - confidence)   # negative number

    for t in range(lookback, n):
        window = returns[t - lookback : t] * 100   # % terms for arch

        try:
            am  = arch_model(window, vol="Garch", p=1, q=1,
                             mean="Zero", dist="normal")
            res = am.fit(disp="off", show_warning=False)

            # 1-step-ahead forecast → variance → vol
            fc       = res.forecast(horizon=1, reindex=False)
            sigma_t  = np.sqrt(fc.variance.values[-1, 0])   # still in % terms
            sigma_t /= 100                                    # back to decimal

            var[t] = -z * sigma_t   # positive loss number

        except Exception:
            # If optimiser fails on a window, fall back to rolling parametric
            sigma_fb = window.std() / 100
            var[t]   = -z * sigma_fb

    return var


print("Computing rolling GARCH VaR — this will take a few minutes...")
print("(Refitting GARCH 2,262 times on 252-day windows)\n")

garch_var_99 = rolling_garch_var(returns, lookback, 0.99)
garch_var_95 = rolling_garch_var(returns, lookback, 0.95)

print("Done.")
valid = ~np.isnan(garch_var_99)
print(f"\nGARCH 99% VaR — rolling stats (post-warmup):")
print(f"  Min  : {garch_var_99[valid].min()*100:.4f}%")
print(f"  Max  : {garch_var_99[valid].max()*100:.4f}%  ← spikes during crises")
print(f"  Mean : {garch_var_99[valid].mean()*100:.4f}%")

## Step 4 — Backtesting GARCH VaR
Same Kupiec + Christoffersen tests as Phase 4.
We expect:
- Fewer exceptions (GARCH widens VaR during stress)
- Lower π11 (exceptions less clustered — GARCH absorbed the regime shift)

In [ ]:
def get_exceptions(returns, rolling_var):
    exc = np.zeros(len(returns), dtype=int)
    valid = ~np.isnan(rolling_var)
    exc[valid] = (returns[valid] < -rolling_var[valid]).astype(int)
    return exc

def kupiec_test(exceptions, confidence, lookback):
    exc    = exceptions[lookback:]
    T, N   = len(exc), exc.sum()
    p_hat  = N / T
    p_star = 1 - confidence
    if N == 0 or N == T:
        return {"N": N, "T": T, "p_hat": p_hat, "LR_uc": np.nan,
                "p_value": np.nan, "reject": True}
    LR_uc = -2 * (
        np.log((1 - p_star) ** (T - N) * p_star ** N)
        - np.log((1 - p_hat) ** (T - N) * p_hat ** N)
    )
    p_value = 1 - chi2.cdf(LR_uc, df=1)
    return {"N": N, "T": T, "p_hat": p_hat, "LR_uc": LR_uc,
            "p_value": p_value, "reject": p_value < 0.05}

def christoffersen_test(exceptions, confidence, lookback):
    exc = exceptions[lookback:].astype(int)
    T   = len(exc)
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        prev, curr = exc[t-1], exc[t]
        if   prev == 0 and curr == 0: n00 += 1
        elif prev == 0 and curr == 1: n01 += 1
        elif prev == 1 and curr == 0: n10 += 1
        elif prev == 1 and curr == 1: n11 += 1
    pi01 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0
    pi11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0
    pi   = (n01 + n11) / T
    eps  = 1e-10
    LR_ind = -2 * (
        np.log(pi + eps) * (n01 + n11) + np.log(1 - pi + eps) * (n00 + n10)
        - np.log(pi01 + eps) * n01 - np.log(1 - pi01 + eps) * n00
        - np.log(pi11 + eps) * n11 - np.log(1 - pi11 + eps) * n10
    )
    LR_ind  = max(LR_ind, 0)
    k       = kupiec_test(exceptions, confidence, lookback)
    LR_uc   = k["LR_uc"] if not np.isnan(k["LR_uc"]) else 0
    LR_cc   = LR_uc + LR_ind
    p_cc    = 1 - chi2.cdf(LR_cc, df=2)
    return {"n00": n00, "n01": n01, "n10": n10, "n11": n11,
            "pi01": pi01, "pi11": pi11,
            "LR_cc": LR_cc, "p_value_cc": p_cc, "reject_cc": p_cc < 0.05}

# ── Run backtests ──
exc_garch_99 = get_exceptions(returns, garch_var_99)
exc_garch_95 = get_exceptions(returns, garch_var_95)

for conf, exc_g in [(0.99, exc_garch_99), (0.95, exc_garch_95)]:
    k  = kupiec_test(exc_g, conf, lookback)
    cr = christoffersen_test(exc_g, conf, lookback)
    print(f"── GARCH {int(conf*100)}% VaR ────────────────────────────────")
    print(f"  Exceptions : {k['N']} / {k['T']}  ({k['p_hat']:.3%})  expected {(1-conf):.1%}")
    print(f"  π01        : {cr['pi01']:.4f}")
    print(f"  π11        : {cr['pi11']:.4f}  (Phase 4 HS was 0.1429)")
    print(f"  Kupiec     : {'PASS ✓' if not k['reject'] else 'FAIL ✗'}  (p={k['p_value']:.4f})")
    print(f"  Christoff  : {'PASS ✓' if not cr['reject_cc'] else 'FAIL ✗'}  (p={cr['p_value_cc']:.4f})")
    print()

## Step 4b — GARCH(1,1) with Student-t Innovations
Normal GARCH still fails Kupiec because z comes from a thin-tailed normal distribution.
Student-t has fatter tails controlled by degrees-of-freedom ν (estimated from data).
The fatter tails push VaR further out → fewer exceptions → closer to the 1% target.

In [ ]:
from scipy.stats import t as student_t

# First fit Student-t GARCH on full sample to inspect ν
am_t = arch_model(
    returns_pct,
    vol="Garch", p=1, q=1,
    mean="Zero",
    dist="t",    # Student-t innovations
)
res_t = am_t.fit(disp="off")

nu    = res_t.params["nu"]         # estimated degrees of freedom
omega_t = res_t.params["omega"]
alpha_t = res_t.params["alpha[1]"]
beta_t  = res_t.params["beta[1]"]

print(f"── GARCH(1,1)-t Parameters ───────────────────────────")
print(f"  ω = {omega_t:.6f}  α = {alpha_t:.4f}  β = {beta_t:.4f}  α+β = {alpha_t+beta_t:.4f}")
print(f"  ν (degrees of freedom) = {nu:.4f}")
print(f"\n  Normal 99% z  : {norm.ppf(0.01):.4f}  (thinner tail)")
print(f"  Student-t 99% z (ν={nu:.1f}): {student_t.ppf(0.01, df=nu):.4f}  (fatter tail)")
print(f"\n  → t-quantile is {abs(student_t.ppf(0.01,df=nu)/norm.ppf(0.01)):.3f}x more conservative than normal")
print(f"\n  Low ν (< 10) confirms fat tails in the data — normal assumption was wrong.")

In [ ]:
def rolling_garch_t_var(
    returns: np.ndarray,
    lookback: int,
    confidence: float,
) -> np.ndarray:
    """
    Rolling GARCH(1,1) VaR with Student-t innovations.
    On each window: fits GARCH-t, estimates ν, gets σ_forecast,
    then VaR = -t_ν(1-c) * σ_forecast.
    """
    n   = len(returns)
    var = np.full(n, np.nan)

    for t in range(lookback, n):
        window = returns[t - lookback : t] * 100

        try:
            am  = arch_model(window, vol="Garch", p=1, q=1,
                             mean="Zero", dist="t")
            res = am.fit(disp="off", show_warning=False)

            nu_t     = res.params["nu"]
            fc       = res.forecast(horizon=1, reindex=False)
            sigma_t  = np.sqrt(fc.variance.values[-1, 0]) / 100

            # Student-t quantile (negative number, fatter tail than normal)
            z_t      = student_t.ppf(1 - confidence, df=nu_t)
            var[t]   = -z_t * sigma_t

        except Exception:
            # Fallback to normal GARCH if t fails
            try:
                am  = arch_model(window, vol="Garch", p=1, q=1,
                                 mean="Zero", dist="normal")
                res = am.fit(disp="off", show_warning=False)
                fc       = res.forecast(horizon=1, reindex=False)
                sigma_t  = np.sqrt(fc.variance.values[-1, 0]) / 100
                var[t]   = -norm.ppf(1 - confidence) * sigma_t
            except Exception:
                sigma_fb = window.std() / 100
                var[t]   = -norm.ppf(1 - confidence) * sigma_fb

    return var


print("Computing rolling GARCH-t VaR — this will take a few minutes...")
garch_t_var_99 = rolling_garch_t_var(returns, lookback, 0.99)
garch_t_var_95 = rolling_garch_t_var(returns, lookback, 0.95)
print("Done.")

In [ ]:
exc_garch_t_99 = get_exceptions(returns, garch_t_var_99)
exc_garch_t_95 = get_exceptions(returns, garch_t_var_95)

# ── Full 4-way comparison table ─────────────────────────────────────────────
rows = []

for name, key in [("Hist Sim (Ph4)", "HS"), ("Parametric (Ph4)", "Parametric"), ("MC (Ph4)", "MC")]:
    k  = kup_99[key]
    cr = chris_99[key]
    rows.append({
        "Method":     name,
        "Exceptions": k["N"],
        "Act. Rate":  f"{k['p_hat']:.3%}",
        "π11":        f"{cr['pi11']:.4f}",
        "Kupiec":     "PASS ✓" if not k["reject"]    else "FAIL ✗",
        "Christoff":  "PASS ✓" if not cr["reject_cc"] else "FAIL ✗",
    })

for name, exc in [("GARCH-Normal", exc_garch_99), ("GARCH-t  ←", exc_garch_t_99)]:
    k  = kupiec_test(exc, 0.99, lookback)
    cr = christoffersen_test(exc, 0.99, lookback)
    rows.append({
        "Method":     name,
        "Exceptions": k["N"],
        "Act. Rate":  f"{k['p_hat']:.3%}",
        "π11":        f"{cr['pi11']:.4f}",
        "Kupiec":     "PASS ✓" if not k["reject"]    else "FAIL ✗",
        "Christoff":  "PASS ✓" if not cr["reject_cc"] else "FAIL ✗",
    })

df = pd.DataFrame(rows).set_index("Method")
print(f"── 99% VaR Full Comparison (T={T}, expected={( 0.01*T):.1f}) ─────────")
print(df.to_string())

In [ ]:
# Visual: all four VaR paths on one plot — calm vs crisis periods
fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(valid_dates, actual_valid, color="grey", linewidth=0.5, alpha=0.6,
        label="Actual return")
ax.plot(valid_dates, -roll_hs_99[lookback:] * 100,
        color="steelblue", linewidth=1.0, alpha=0.8, label="HS (Ph4)")
ax.plot(valid_dates, -garch_var_99[lookback:] * 100,
        color="darkorange", linewidth=1.0, alpha=0.8, label="GARCH-Normal")
ax.plot(valid_dates, -garch_t_var_99[lookback:] * 100,
        color="tomato", linewidth=1.2, label="GARCH-t")

ax.axhline(0, color="black", linewidth=0.5)
ax.set_ylabel("Return / VaR (%)")
ax.set_title("99% VaR: Static HS vs GARCH-Normal vs GARCH-t\n"
             "GARCH-t should be the widest line — most conservative in calm AND crisis")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Save GARCH-t results
data.update({
    "garch_t_var_99": garch_t_var_99,
    "garch_t_var_95": garch_t_var_95,
    "exc_garch_t_99": exc_garch_t_99,
    "exc_garch_t_95": exc_garch_t_95,
    "garch_t_params": {"omega": omega_t, "alpha": alpha_t,
                       "beta": beta_t,   "nu": nu},
})
with open("portfolio_data.pkl", "wb") as f:
    pickle.dump(data, f)
print("Saved GARCH-t results to portfolio_data.pkl")

## Step 5 — Head-to-Head Comparison: GARCH vs Phase 4 Methods

In [ ]:
c = 0.99
T = len(returns) - lookback

rows = []

# Phase 4 methods
for name, key in [("Hist Sim (Ph4)", "HS"), ("Parametric (Ph4)", "Parametric"), ("MC (Ph4)", "MC")]:
    k  = kup_99[key]
    cr = chris_99[key]
    rows.append({
        "Method":      name,
        "Exceptions":  k["N"],
        "Act. Rate":   f"{k['p_hat']:.3%}",
        "π11":         f"{cr['pi11']:.4f}",
        "Kupiec":      "PASS ✓" if not k["reject"]    else "FAIL ✗",
        "Christoff":   "PASS ✓" if not cr["reject_cc"] else "FAIL ✗",
    })

# GARCH
k_g  = kupiec_test(exc_garch_99, c, lookback)
cr_g = christoffersen_test(exc_garch_99, c, lookback)
rows.append({
    "Method":    "GARCH(1,1)  ←",
    "Exceptions": k_g["N"],
    "Act. Rate":  f"{k_g['p_hat']:.3%}",
    "π11":        f"{cr_g['pi11']:.4f}",
    "Kupiec":     "PASS ✓" if not k_g["reject"]    else "FAIL ✗",
    "Christoff":  "PASS ✓" if not cr_g["reject_cc"] else "FAIL ✗",
})

df = pd.DataFrame(rows).set_index("Method")
print(f"── 99% VaR Backtest Comparison (T = {T} days) ──────────────")
print(f"   Expected exceptions: {(1-c)*T:.1f}  ({(1-c):.1%})\n")
print(df.to_string())

## Step 6 — Visual Comparison: Rolling VaR Paths

In [ ]:
valid_dates  = dates[lookback:]
actual_valid = returns[lookback:] * 100
roll_hs_99   = all_bt[0.99]["roll_hs"]
exc_hs_99    = all_bt[0.99]["exc_hs"]

fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# ── Panel 1: Historical Simulation (Phase 4 baseline) ──────────────────────
ax = axes[0]
ax.plot(valid_dates, actual_valid, color="grey", linewidth=0.5, alpha=0.7,
        label="Actual return")
ax.plot(valid_dates, -roll_hs_99[lookback:] * 100, color="steelblue",
        linewidth=1.2, label="HS 99% VaR (Phase 4)")

hs_exc_mask = exc_hs_99[lookback:] == 1
ax.scatter(valid_dates[hs_exc_mask], actual_valid[hs_exc_mask],
           color="red", s=12, zorder=5, label=f"Exceptions ({hs_exc_mask.sum()})")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_ylabel("Return (%)")
ax.set_title("Historical Simulation 99% VaR — Static Window (Phase 4 baseline)")
ax.legend(fontsize=8, loc="lower right")

# ── Panel 2: GARCH ──────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(valid_dates, actual_valid, color="grey", linewidth=0.5, alpha=0.7,
        label="Actual return")
ax.plot(valid_dates, -garch_var_99[lookback:] * 100, color="darkorange",
        linewidth=1.2, label="GARCH(1,1) 99% VaR")

garch_exc_mask = exc_garch_99[lookback:] == 1
ax.scatter(valid_dates[garch_exc_mask], actual_valid[garch_exc_mask],
           color="red", s=12, zorder=5, label=f"Exceptions ({garch_exc_mask.sum()})")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_ylabel("Return (%)")
ax.set_title("GARCH(1,1) 99% VaR — Dynamic Volatility (Phase 7)")
ax.legend(fontsize=8, loc="lower right")

plt.suptitle("Rolling 99% VaR: Static vs GARCH — Exception Comparison",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# What to look for:
# GARCH line (orange) should widen dramatically in March 2020 and late 2022
# HS line (blue) widens slowly as bad days accumulate in the 252-day window
# GARCH should have fewer red dots concentrated in the same cluster spots

In [ ]:
# Save GARCH results to portfolio_data.pkl for use in dashboard / future phases
garch_results = {
    "garch_var_99":   garch_var_99,
    "garch_var_95":   garch_var_95,
    "exc_garch_99":   exc_garch_99,
    "exc_garch_95":   exc_garch_95,
    "garch_params":   {"omega": omega, "alpha": alpha, "beta": beta},
}

data.update(garch_results)

with open("portfolio_data.pkl", "wb") as f:
    pickle.dump(data, f)

print("portfolio_data.pkl updated with Phase 7 GARCH results.")
print(f"\nGARCH parameters saved:")
print(f"  ω = {omega:.6f}  α = {alpha:.4f}  β = {beta:.4f}  α+β = {alpha+beta:.4f}")